In [1]:
import pandas as pd

import wrds

ModuleNotFoundError: No module named 'pandas'

In [ ]:
filepath = "AGG_holdings_clean.csv"
ishare_df = pd.read_csv(filepath, skiprows=9, header=0)
cols_to_drop = ['Location', 'Exchange', 'Currency','Yield to Call (%)', 'Asset Class']
ishare_df = ishare_df.drop(columns=cols_to_drop)
ishare_df.groupby('Sector')['Weight (%)'].sum().sort_values(ascending=False).head(12) 

Sector
Treasury                   46.85412
MBS Pass-Through           22.86333
Industrial                 14.21748
Financial Institutions      7.99986
Utility                     2.48722
CMBS                        1.39747
Sovereign                   1.08776
Supranational               0.91401
Cash and/or Derivatives     0.66841
Local Authority             0.58526
Agency                      0.50100
ABS                         0.42405
Name: Weight (%), dtype: float64

In [9]:
db = wrds.Connection(wrds_username="hauvi")
# Get the portfolio number from CRSP's metadata for AGG
agg_query = """
    SELECT * FROM crsp.fund_names WHERE ticker = 'AGG'
    """ 
agg_wrds_df = db.raw_sql(agg_query)
agg_wrds_df.sort_values(by='chgdt', ascending=False, inplace=True) 

# Get the most recent portfolio number and query the holdings specifically
portno = int(agg_wrds_df.sort_values("chgdt")["crsp_portno"].iloc[-1]) # get the most recent AGG portfolio (june 30th) idk how we can get more recent ones but whatever
col_to_keep = [ 'security_name', 'cusip', 'crsp_company_key', 'ticker', 'coupon', 'maturity_dt', 'nbr_shares', 'market_val', 'percent_tna','security_rank']
holdings = db.raw_sql(f"""
    select {', '.join(col_to_keep)} from crsp.holdings
    where crsp_portno = {portno}
      and report_dt = '2026-06-30'
""")

Loading library list...
Done


In [28]:
db.get_table(library='crsp', table='tfz_dly', obs=10)

,kytreasno,kycrspid,caldt,tdbid,tdask,tdnomprc,tdnomprc_flg,tdsourcr,tdaccint,tdretnua,tdyld,tdduratn,tdpubout,tdtotout,tdpdint,tdidxratio,tdidxratio_flg
0,200001.0,19610622.800000,1961-06-14,99.956667,99.961111,99.958889,M,R,0.0,<NA>,0.000051,8.0,<NA>,3503.0,0.0,<NA>,<NA>
1,200001.0,19610622.800000,1961-06-15,99.963056,99.966944,99.965,M,R,0.0,0.000061,0.00005,7.0,<NA>,3503.0,0.0,<NA>,<NA>
2,200001.0,19610622.800000,1961-06-16,99.965833,99.971667,99.96875,M,R,0.0,0.000038,0.000052,6.0,<NA>,3503.0,0.0,<NA>,<NA>
3,200001.0,19610622.800000,1961-06-19,99.981667,99.984583,99.983125,M,R,0.0,0.000144,0.000056,3.0,<NA>,3503.0,0.0,<NA>,<NA>
4,200001.0,19610622.800000,1961-06-20,99.9875,99.989722,99.988611,M,R,0.0,0.000055,0.000057,2.0,<NA>,3503.0,0.0,<NA>,<NA>
5,200002.0,19610623.400000,1961-06-14,99.9525,99.9575,99.955,M,R,0.0,<NA>,0.00005,9.0,<NA>,1595.0,0.0,<NA>,<NA>
6,200002.0,19610623.400000,1961-06-15,99.957778,99.962222,99.96,M,R,0.0,0.00005,0.00005,8.0,<NA>,1595.0,0.0,<NA>,<NA>
7,200002.0,19610623.400000,1961-06-16,99.963056,99.965972,99.964514,M,R,0.0,0.000045,0.000051,7.0,<NA>,1595.0,0.0,<NA>,<NA>
8,200002.0,19610623.400000,1961-06-19,99.976667,99.98,99.978333,M,R,0.0,0.000138,0.000054,4.0,<NA>,1595.0,0.0,<NA>,<NA>
9,200002.0,19610623.400000,1961-06-20,99.9825,99.984583,99.983542,M,R,0.0,0.000052,0.000055,3.0,<NA>,1595.0,0.0,<NA>,<NA>


In [ ]:
# Daily returns of the AGG ETF I assume?
# can also query from crsp.daily_nav_ret 
# monthly: monthly_returns, monthly_tna_ret_nva
query = """
    SELECT * FROM crsp.daily_returns
    where crsp_fundno = 16417
"""
db.raw_sql(query).sort_values(by='caldt', ascending=False)

,crsp_fundno,caldt,dret
5728,16417.0,2026-06-30,-0.004141
5727,16417.0,2026-06-29,0.000495
5726,16417.0,2026-06-26,0.000889
5725,16417.0,2026-06-25,0.000575
5724,16417.0,2026-06-24,0.004717
...,...,...,...
4,16417.0,2003-09-26,0.003447
3,16417.0,2003-09-25,0.001282
2,16417.0,2003-09-24,0.003563
1,16417.0,2003-09-23,0.001189


****